In [1]:
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")

In [2]:
%%microblaze base.PMODA
#include <stdint.h>
#include <pyprintf.h>
#include "spi.h"

static char tx_buff;
static char rx_buff;

unsigned int DATA_BYTE_SIZE = 1;
unsigned int SCLK_PIN = 0;
unsigned int MISO_PIN = 1;
unsigned int MOSI_PIN = 2;
unsigned int CS_PIN = 3;

spi spi_dev;

//initialize SPI device
int spi_init(void){
    spi_dev = spi_open(SCLK_PIN, MISO_PIN, MOSI_PIN, CS_PIN);
    return -1;
}

// get spi data 
uint8_t get_spi_data(){
    spi_transfer(spi_dev, &tx_buff, &rx_buff, DATA_BYTE_SIZE);
    uint8_t rx_data = (uint8_t)rx_buff;
    
    return rx_data;
}


In [6]:
# SPI Helper functions
def get_laser_state(data_byte):
    return data_byte >> 7

def get_motor_cmd(data_byte):
    return data_byte & 0x0F

In [ ]:
# test
# loop through and get data being sent
delete = spi_init()
while True:
    data = int(get_spi_data())
    laser = get_laser_state(data)
    cmd = get_motor_cmd(data)
    if cmd > 0 and cmd < 255:
        print(f"Command Received: {cmd}\n")
        print(f"Laser State: {laser}")


In [ ]:
car = RC_Car(False, logging.DEBUG)
car.laser.shoot()
time.sleep(5)
delete = spi_init()
while True:
    data = int(get_spi_data())
    laser = get_laser_state(data)
    cmd = get_motor_cmd(data)
    if cmd > 0 and cmd < 255:
        print(f"Command Received: {cmd}\n")
        print(f"Laser State: {laser}")
    car.move(cmd)
    time.sleep(0.1)